In [ ]:
import os
import glob
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import scipy.linalg as la

# ==============================================================================
# 1. DATASET PARAMÉTRICO ULTRA RÁPIDO (PROPRIEDADES TENSORIAIS GLOBAIS)
# ==============================================================================
class FastRotorStateDataset(Dataset):
    def __init__(self, pasta_dados):
        super().__init__()
        self.samples = []
        
        padrao_busca = os.path.join(pasta_dados, "*.pkl")
        arquivos_encontrados = glob.glob(padrao_busca)
        
        print(f"📂 Carregando base de dados rápida com física de autovalores...")
        for caminho_arquivo in arquivos_encontrados:
            with open(caminho_arquivo, "rb") as f:
                dados_lote = pickle.load(f)
            if isinstance(dados_lote, list):
                self.samples += dados_lote
        print(f"✅ Sucesso: {len(self.samples)} amostras carregadas.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        M = sample['Matriz_M']
        K = sample['Matriz_K']
        C = sample['Matriz_C']
        
        # Se houver matriz giroscópica salva, soma ela ao amortecimento (C_total = C + W*G)
        # Caso não use síncrono ou não tenha 'Matriz_G', ele usa a C pura de filme fluido
        C_total = C + sample.get('Matriz_G', 0) if 'Matriz_G' in sample else C
        
        n = M.shape[0]
        
        # 1. MONTAGEM DAS MATRIZES DE ESTADO DE DUNCAN (2n x 2n)
        # B = [[C_total, M], [M, 0]]
        B_top = np.hstack([C_total, M])
        B_bot = np.hstack([M, np.zeros((n, n))])
        B_mat = np.vstack([B_top, B_bot])
        
        # A = [[-K, 0], [0, M]]
        A_top = np.hstack([-K, np.zeros((n, n))])
        A_bot = np.hstack([np.zeros((n, n)), M])
        A_mat = np.vstack([A_top, A_bot])
        
        # 2. RESOLUÇÃO DE det(A - lambda * B) = 0
        try:
            autovalores_brutos = la.eigvals(A_mat, B_mat)
            autovalores_validos = autovalores_brutos[np.isfinite(autovalores_brutos)]
            
            # Ordena pelo menor módulo absoluto (modos mais baixos)
            indices_ordenados = np.argsort(np.abs(autovalores_validos))
            primeiros_autovalores = autovalores_validos[indices_ordenados[:3]]
            
            if len(primeiros_autovalores) < 3:
                primeiros_autovalores = np.pad(primeiros_autovalores, (0, 3 - len(primeiros_autovalores)), 'constant')
                
        except la.LinAlgError:
            primeiros_autovalores = np.zeros(3, dtype=complex)
        
        # 💡 CORREÇÃO DE ESCALA: Passando os autovalores para Log para casar com a FRF
        features_estado = []
        for lmbda in primeiros_autovalores:
            # Parte Real (Amortecimento): tipicamente negativa, pegamos o abs antes do log
            log_real = np.log10(np.abs(lmbda.real) + 1e-5)
            # Parte Imaginária (Frequência): frequência natural amortecida em log
            log_imag = np.log10(np.abs(lmbda.imag) + 1e-5)
            
            features_estado.append(log_real)
            features_estado.append(log_imag)
            
        features_estado = np.array(features_estado)
        
        # 3. EXTRAÇÃO DOS MANCAIS (Comportamento Local Log)
        k_diag = np.diag(K)
        c_diag = np.diag(C)
        k_mancais = np.log10(k_diag[k_diag > 1e3] + 1e-5)
        c_mancais = np.log10(c_diag[c_diag > 1e-1] + 1e-5)
        
        if len(k_mancais) == 0: k_mancais = np.log10(k_diag[:4] + 1e-5)
        if len(c_mancais) == 0: c_mancais = np.log10(c_diag[:4] + 1e-5)
        
        # GDLs de ensaio
        gdl_in = float(sample['Node_in'])
        gdl_out = float(sample['Node_out'])
        
        # 4. CONCATENAÇÃO FINAL DAS ENTRADAS
        # Vetor de entrada fixo: Mancais + Assinatura exata do espaço de estados (6) + GDLs (2)
        x_design = np.concatenate([k_mancais, c_mancais, features_estado, [gdl_in, gdl_out]])
        
        frf_log = np.log10(np.abs(sample['FRF']) + 1e-15)
        
        return {
            'x': torch.tensor(x_design, dtype=torch.float),
            'y': torch.tensor(frf_log, dtype=torch.float)
        }

# ==============================================================================
# 2. SEU FOURIER FEATURE VITORIOSO ORIGINAL
# ==============================================================================
class PureFourierFeatureLayer(nn.Module):
    def __init__(self, input_dim, mapping_size=256, scale=10.0):
        super().__init__()
        B = torch.randn(input_dim, mapping_size) * scale
        self.register_buffer('B', B)

    def forward(self, x):
        projection = 2 * np.pi * torch.matmul(x, self.B)
        return torch.cat([torch.sin(projection), torch.cos(projection)], dim=-1)

# ==============================================================================
# 3. ARQUITETURA MLP PROFUNDA EQUIPADA COM SEU FOURIER FEATURE
# ==============================================================================
class FastRotorFourierSurrogate(nn.Module):
    def __init__(self, input_dim, mapping_size=256, num_frequencies=1000):
        super().__init__()
        
        self.fourier = PureFourierFeatureLayer(input_dim=input_dim, mapping_size=mapping_size, scale=10.0)
        fourier_dim = mapping_size * 2
        
        self.mlp = nn.Sequential(
            nn.Linear(fourier_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            
            nn.Linear(1024, num_frequencies)
        )
        
    def forward(self, x):
        return self.mlp(self.fourier(x))

# ==============================================================================
# 4. EXECUÇÃO DO TREINAMENTO DE ALTA VELOCIDADE
# ==============================================================================
pasta_externa = r"C:\\Users\\Cliente\\Desktop\\Pastas\\Códigos\\surrogate_rotor_model\\dados_IA"

dataset = FastRotorStateDataset(pasta_dados=pasta_externa)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

input_dim_real = dataset[0]['x'].shape[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Dispositivo: {device} | Dimensão dos Inputs Enxutos: {input_dim_real}")

model = FastRotorFourierSurrogate(input_dim=input_dim_real, mapping_size=256).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

print("🚀 Treinando na velocidade da luz...")
for epoch in range(1, 301):
    model.train()
    total_loss = 0
    for batch in loader:
        x_batch = batch['x'].to(device)
        y_batch = batch['y'].to(device)
        
        optimizer.zero_grad()
        out = model(x_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
        
    epoch_loss = total_loss / len(dataset)
    scheduler.step(epoch_loss)
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"   Época {epoch:03d}/300 | Loss Estável: {epoch_loss:.6f}")

print("✅ Concluído! O tempo de treino voltou ao normal.")

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Garante o modo de avaliação da rede
model.eval()

# 1. Sorteia um índice aleatório dentro do dataset
idx_amostra = random.randint(0, len(dataset) - 1)
amostra = dataset[idx_amostra]

# 2. Reconstroi o vetor único 'x' dependendo de como o seu dataset entregou os dados
if 'x' in amostra:
    # Se o dataset antigo/atual na memória entrega 'x' pronto
    x_input = amostra['x'].unsqueeze(0).to(device)
else:
    # Se por acaso o dataset já mudou chaves, junta de volta para o modelo de 1 argumento
    x_input = torch.cat([amostra['x_struct'], amostra['x_space']]).unsqueeze(0).to(device)

# 3. Executa a inferência passando apenas o argumento único exigido pelo seu modelo atual
with torch.no_grad():
    y_pred = model(x_input).cpu().squeeze(0).numpy()

y_real = amostra['y'].numpy()
speed_range = dataset.samples[idx_amostra]['Speed_range']

# Localiza os picos reais para fins de referência visual
indices_picos_reais, _ = find_peaks(y_real, height=-6.5, distance=50)

# 4. MONTAGEM DOS GRÁFICOS (FRF Log10 + Erro Absoluto em Escala Log)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True, 
                               gridspec_kw={'height_ratios': [2, 1]})

# --- Subplot 1: Curvas de Resposta (FRF) ---
ax1.plot(speed_range, y_real, label='ROSS (Real)', color='black', lw=2, ls='--')
ax1.plot(speed_range, y_pred, label='IA Predict', color='blue', lw=2)
if len(indices_picos_reais) > 0:
    ax1.scatter(speed_range[indices_picos_reais], y_real[indices_picos_reais], 
                color='red', s=40, zorder=5, label='Picos Reais')

ax1.set_title(f'Validação Paramétrica - Amostra #{idx_amostra}', fontsize=12, fontweight='bold')
ax1.set_ylabel('Magnitude da FRF (Módulo Log10)', fontsize=11)
ax1.grid(True, which="both", alpha=0.3)
ax1.legend(fontsize=10)

# --- Subplot 2: Erro Absoluto em ESCALA LOG ---
erro_log = np.abs(y_pred - y_real)
ax2.plot(speed_range, erro_log, color='darkviolet', lw=1.5, label='|Δ Log10|')
ax2.set_xlabel('Frequência / Rotação (rad/s)', fontsize=11)
ax2.set_ylabel('Erro Absoluto (Δ Log10)', fontsize=11)
ax2.grid(True, which="both", alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# 5. EXIBIÇÃO DOS PARÂMETROS DA AMOSTRA ATUAL
dados_crus = dataset.samples[idx_amostra]
print(f"📊 Detalhes do Ensaio Sorteado:")
print(f"   -> Nó de Impacto (Entrada): {dados_crus['Node_in']}")
print(f"   -> Nó do Sensor (Saída): {dados_crus['Node_out']}")